import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tr = TestResult()

In [1]:
import sys
sys.path.insert(0, '.')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

tr = TestResult()

✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200


In [ ]:
# Cell 2: Bootstrap — create watchdog principal + resources using root token
#
# Run mode A (first time / fresh env):
#   ensure_watchdog_setup() creates everything, prints clientSecret
#
# Run mode B (401 unauthorized_client — stale/lost secret):
#   reset_watchdog_principal() deletes + recreates principal, prints new secret
#
# After either run: copy the printed clientSecret into WATCHDOG_SECRET below.

# ── Choose one ─────────────────────────────────────────────────
client_id, new_secret = ensure_watchdog_setup()
# client_id, new_secret = reset_watchdog_principal()  # ← use this on 401

# ── Paste saved secret here (only needed when new_secret is None) ──
WATCHDOG_SECRET = new_secret or "<paste-saved-secret-here>"

print(f"\nWATCHDOG_SECRET ready: {'✅' if WATCHDOG_SECRET != '<paste-saved-secret-here>' else '❌ not set — paste secret above'}")

In [3]:
WATCHDOG_CLIENT_ID = "99fc1f19d76f5ee7"
WATCHDOG_SECRET = "4a50f0a84b9752c391fa2700d4425296"

In [5]:
# T01: OAuth2 token acquisition (watchdog principal, default realm)
print("T01: OAuth2 token acquisition")
try:
    token = get_watchdog_token(WATCHDOG_CLIENT_ID, WATCHDOG_SECRET)
    tr.record("T01_auth", True, "token acquired")
except AssertionError as e:
    tr.record("T01_auth", False, str(e))
    print("⛔ Cannot continue without token")
    raise

T01: OAuth2 token acquisition
  ✅ T01_auth: token acquired


In [6]:
# T02: Watchdog catalog accessible (via catalog API)
# Note: management /api/management/v1/catalogs requires service_admin privilege.
# watchdog-principal only has catalog-level access — use catalog API instead.
print("\nT02: Watchdog catalog accessible")
r = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces",
    headers=h(token)
)
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    ns_list = r.json().get("namespaces", [])
    detail = f"catalog reachable, namespaces: {ns_list}"
tr.record("T02_catalog_accessible", passed, detail)


T02: Watchdog catalog accessible
  ✅ T02_catalog_accessible: catalog reachable, namespaces: []


In [7]:
# T03: Namespace CRUD (within watchdog-catalog)
print("\nT03: Namespace CRUD")

r_create = create_namespace(catalog=WATCHDOG_CATALOG, ns=WATCHDOG_NAMESPACE, token=token)
passed_create = r_create.status_code in (200, 201)
tr.record("T03a_namespace_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T03b_namespace_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T03_namespace_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")


T03: Namespace CRUD
  ✅ T03a_namespace_create: HTTP 200
  ✅ T03b_namespace_read: HTTP 200
  ✅ T03_namespace_crud: create + read OK


In [8]:
# T04: Table CRUD
print("\nT04: Table CRUD")

r_create = requests.post(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables",
    headers=h(token),
    json={
        "name": WATCHDOG_TABLE,
        "location": f"s3a://data-catalog-bucket/{WATCHDOG_CATALOG}/{WATCHDOG_NAMESPACE}/{WATCHDOG_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",    "type": "long",   "required": True},
                {"id": 2, "name": "value", "type": "string", "required": False}
            ]
        }
    }
)
passed_create = r_create.status_code in (200, 201)
tr.record("T04a_table_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T04b_table_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T04_table_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")


T04: Table CRUD
  ✅ T04a_table_create: HTTP 200
  ✅ T04b_table_read: HTTP 200
  ✅ T04_table_crud: create + read OK


In [9]:
# T05: RBAC enforcement (negative test)
# Try to delete a catalog using the watchdog token — must return 403.
# watchdog-principal only has CATALOG_MANAGE_CONTENT on watchdog-catalog,
# so any DELETE on other catalogs or the catalog itself via management API
# should be forbidden.
print("\nT05: RBAC enforcement — watchdog must NOT delete catalogs via management API")

# Use watchdog token (not root) — must be rejected
r = requests.delete(
    f"{BASE_MGMT}/catalogs/non-existent-catalog",
    headers=h(token)
)
# Polaris returns 403 (no privilege) or 404 depending on implementation
# Either is acceptable — what matters is NOT 200/204
passed = r.status_code in (403, 404, 401)
tr.record("T05_rbac_enforcement", passed,
    f"HTTP {r.status_code} ({'RBAC working' if passed else 'RBAC BROKEN — watchdog has too much access'})")


T05: RBAC enforcement — watchdog must NOT delete catalogs via management API
  ✅ T05_rbac_enforcement: HTTP 404 (RBAC working)


In [10]:
# T06: Health correlation — detect entity_version mismatch
print("\nT06: Health correlation")
r_health = requests.get(f"{POLARIS_URL}/q/health")
health_up = r_health.status_code == 200 and r_health.json().get("status") == "UP"
write_failed = not tr.results.get("T03_namespace_crud", {}).get("passed", True)

if write_failed and health_up:
    tr.record("T06_health_correlation", False,
        "⚠️  entity_version mismatch suspected — health=UP but write failed")
    print()
    print("⚠️  ENTITY VERSION MISMATCH DETECTED")
    print("   Fix (ask system engineer to run):")
    print("     kubectl rollout restart deployment/benchmarks-polaris -n datahub-hynix")
    print("   DO NOT restart PostgreSQL")
else:
    tr.record("T06_health_correlation", True,
        f"health={'UP' if health_up else 'DOWN'} write={'OK' if not write_failed else 'FAIL'}")


T06: Health correlation
  ✅ T06_health_correlation: health=DOWN write=OK


In [11]:
# Cleanup — only watchdog's own table + namespace (not the whole catalog)
print("\nCleanup...")
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
print("  table + namespace deleted (catalog + principal kept for next run)")


Cleanup...
  table + namespace deleted (catalog + principal kept for next run)


In [12]:
# Summary
print()
all_passed = tr.summary()
print()
if all_passed:
    print("✅ Polaris fully operational")
else:
    print("❌ Issues detected — check failed tests above")
    print()
    print("Common fixes:")
    print("  (this notebook has no kubectl access — report below to system engineer)")
    print("  T01 fail: ask to check pod status: kubectl get pods -n datahub-hynix | grep polaris")
    print("  T03 fail + T06 warn: ask to restart: kubectl rollout restart deployment/benchmarks-polaris")
    print("  T05 fail: CHECK RBAC IMMEDIATELY — privileges too broad")



Result: 10/10 passed

✅ Polaris fully operational
